# 02 — Embedding Geometry

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/forge-tokenizer/blob/main/notebooks/embedding_geometry.ipynb)

Build tiny distributional embeddings from a local corpus, inspect nearest neighbors, and connect embeddings to unembedding and sampling.

**Runtime:** CPU is fine. No GPU required.

---

## 1. Install & Clone

In [ ]:
!pip install -q numpy matplotlib pytest tqdm regex
%cd /content
!test -d forge-tokenizer || git clone https://github.com/marcoharuni/forge-tokenizer.git
%cd /content/forge-tokenizer
!git pull --ff-only
!pip install -q -e .

## 2. Imports

In [ ]:
from pathlib import Path
import numpy as np
import regex

from forge_tokenizer.embeddings import (
    analogy,
    build_cooccurrence,
    cosine_similarity,
    embedding_parameter_count,
    nearest_neighbors,
    one_hot,
    ppmi_matrix,
    svd_embeddings,
)
from forge_tokenizer.logit_lens import format_layer_predictions, run_simulated_logit_lens
from forge_tokenizer.unembedding import logits_to_probs, sample_from_logits, stable_softmax
from forge_tokenizer.visualization import plot_embedding_neighbors_2d, plot_temperature_distributions

print('forge-tokenizer ready')

## 3. One-Hot Vectors And Parameter Counts

Token IDs become vectors. In a real model, the embedding table learns dense vectors from training.

In [ ]:
print('one_hot(3, 8):', one_hot(3, 8))
print(embedding_parameter_count(vocab_size=50_000, d_model=768, tied=False))
print(embedding_parameter_count(vocab_size=50_000, d_model=768, tied=True))

## 4. Build A Tiny Co-Occurrence Matrix

This is a small PPMI/SVD demonstration from the included corpus. No pretrained embeddings are used.

In [ ]:
text = Path('data/tiny_corpus.txt').read_text(encoding='utf-8').lower()
sentences = []

for line in text.splitlines():
    words = regex.findall(r'\p{L}+', line)
    if words:
        sentences.append(words)

cooc, word_to_idx, idx_to_word = build_cooccurrence(sentences, window_size=2)
print('Sentences:', len(sentences))
print('Vocabulary:', len(word_to_idx))
print('Co-occurrence shape:', cooc.shape)

## 5. PPMI And SVD Embeddings

In [ ]:
ppmi = ppmi_matrix(cooc)
embeddings = svd_embeddings(ppmi, dim=8)

print('PPMI shape:', ppmi.shape)
print('Embedding shape:', embeddings.shape)
print('Cosine self-check:', cosine_similarity(embeddings[0], embeddings[0]))

## 6. Nearest Neighbors

In [ ]:
query = 'tokenization' if 'tokenization' in word_to_idx else next(iter(word_to_idx))
print('Query:', query)

for word, score in nearest_neighbors(embeddings, word_to_idx, idx_to_word, query, k=10):
    print(f'{word:16s} {score: .3f}')

## 7. Tiny Analogy Probe

On a tiny corpus this is only a mechanism demo, not a semantic benchmark.

In [ ]:
required = ['text', 'numbers', 'tokens']
if all(word in word_to_idx for word in required):
    print('text is to numbers as tokens is to ?')
    print(analogy(embeddings, word_to_idx, idx_to_word, 'text', 'numbers', 'tokens', k=5))
else:
    print('Tiny corpus does not contain every analogy word.')

## 8. Plot The First Two Dimensions

In [ ]:
labels = [idx_to_word[i] for i in range(min(24, len(idx_to_word)))]
path = plot_embedding_neighbors_2d(embeddings[:len(labels), :2], labels, 'generated/notebook_embedding_neighbors.png')
path

## 9. Unembedding, Temperature, And Sampling

The same vector-space story ends at logits, probabilities, and token sampling.

In [ ]:
logits = np.array([1000.0, 1001.0, 1002.0])
print('stable softmax:', stable_softmax(logits))
print('top sample:', sample_from_logits(logits, temperature=0.7, seed=0))

path = plot_temperature_distributions(np.array([0.2, 1.1, 2.4, -0.8, 0.0]), 'generated/notebook_temperature.png')
path

## 10. Simulated Logit Lens

This is simulated with random matrices. It is not a trained language model.

In [ ]:
rows = run_simulated_logit_lens(seed=7)
print(format_layer_predictions(rows))

---

**Back →** [Tokenizer Playground](tokenizer_playground.ipynb)